# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access the main metadata fields
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")
print(f"Identifier: {meta.identifier}")
print(f"Authors: {getattr(meta, 'author', [])}")
print(f"Published on: {getattr(meta, 'datePublished', None)}")
print(f"License: {getattr(meta, 'license', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Below we enumerate all record sets in the dataset, list their `@id`s, their available fields, and print out a sample record for each.

In [ ]:
# List all record sets by their @id
print('Available record sets:')
record_sets = dataset.record_sets()
for rs in record_sets:
    print(f"Record set name: {rs.name} | @id: {rs.id}")
    print('  Fields:')
    for field in rs.fields:
        print(f"   - {field.name} (@id: {field.id})")
    # Print one sample record if possible
    sample_records = list(dataset.records(record_set=rs.id))
    if sample_records:
        print('  Sample record:')
        # Only print the first record keys/values
        for k, v in sample_records[0].items():
            print(f"    {k}: {v}")
    print('-' * 50)

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for further analysis.

We extract all main record sets by their `@id`, storing the results in a dictionary.

In [ ]:
# Collect all record set @ids
record_sets = dataset.record_sets()

# Store dataframes for each record set
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded DataFrame for record set '{rs.name}' (@id: {rs.id}) with shape {df.shape}")

# For demonstration, pick the first record set for head/columns display
if record_sets:
    first_rs_id = record_sets[0].id
    print(f"\nColumns in record set '{record_sets[0].name}' (@id: {first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. Reference all fields and record sets by their `@id`.

_The following code demonstrates filtering on a numeric field, normalization, and optional grouping (if a suitable group field exists)._

In [ ]:
# For example, use the fields of the first record set
if record_sets:
    rs = record_sets[0]
    df = dataframes[rs.id]
    print(f"Working with record set: {rs.name} (@id: {rs.id})")

    # Identify numeric fields by inspecting the schema types if present
    numeric_field_id = None
    for field in rs.fields:
        # We look for Float or Integer type
        if hasattr(field, 'data_type') and field.data_type:
            if 'Float' in field.data_type or 'Integer' in field.data_type or 'Number' in field.data_type:
                numeric_field_id = field.id
                print(f"Numeric field identified: {field.name} (@id: {field.id})")
                break
    # Fallback: use the first column that is numeric in the DataFrame
    if not numeric_field_id:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                print(f"Numeric field inferred from data: {numeric_field_id}")
                break
    if numeric_field_id and numeric_field_id in df.columns:
        # Use 10th percentile as a threshold for filtering
        threshold = df[numeric_field_id].quantile(0.10) if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field (e.g., a categorical or object column)
        group_field = None
        for field in rs.fields:
            if field.id != numeric_field_id and field.id in df.columns:
                # Choose a likely categorical field (string type)
                if pd.api.types.is_object_dtype(df[field.id]):
                    group_field = field.id
                    print(f"Grouping field identified: {group_field}")
                    break
        if group_field:
            grouped = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped {numeric_field_id} mean by {group_field}:")
            print(grouped.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets available in the dataset.")

## 5. Visualization
Visualize data distributions or field relationships using matplotlib or seaborn. For demonstration, we use the (inferred) numeric field and one group field if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure prior EDA section identified these variables
if 'filtered_df' in locals() and numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping field available, show boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No filtered data or numeric/group fields available for plotting.')

## 6. Conclusion
We have loaded, explored, and visualized the FAIR² dataset using `mlcroissant` by referencing all elements using their `@id`. This approach supports transparent, reproducible data science workflows that are compatible with FAIR (Findable, Accessible, Interoperable, Reusable) data principles.

**Key observations:**
- The dataset contains ordered logistic regression results regarding knowledge adoption in rangeland management among pastoral communities in Northern Kenya.
- Structured exploration via record set and field `@id`s enables robust, standards-based analytics.
- Data normalization and grouping facilitate deeper analysis; visualization aids interpretation.

_Refer to the dataset's schema for all data element `@id` references when building further pipelines or analytics._